# FastH3 · ComfyUI · Colab L4

另有獨立 [ComfyUI workspace notebook](ComfyUI_Colab.ipynb) 與[操作說明](../doc/COMFYUI_WORKSPACE.md)，提供 H3 Continuum／實驗性原生 Ref2VA-VSA 的安裝、UI 與 API 入口；預設不下載模型、不生成，尚未做 GPU 驗收。

手動選擇 **L4 GPU、標準 RAM**。首次執行 A 區，之後每片只執行 B 區。只支援文字生成有聲影片 (T2VA)，預設 5 秒、864×480、24 fps；示範不需外部素材。

使用內嵌固定版本工具與共同 ComfyUI API graph。預設不加 LoRA；可在 JSON 填相容 adapter 的固定 revision 或本地 path 與 strength，不會自動套用 Turbo。A 區沿用已啟動 server、安裝與磁碟快取；ComfyUI 依需要載入／offload，不保證全部權重常駐 VRAM。

## A. 初始化一次


In [ ]:
# @title 1. FastH3 L4 設定（新的 runtime 執行一次）
BACKEND = 'FastH3 | ComfyUI L4 (T2VA)'
FAST_H3_LORAS_JSON = "[]" # @param {type:"string"}
# Optional compatible adapters only: [{"repo":"owner/repo","file":"adapter.safetensors","revision":"full_commit","strength":1.0}]
# Or an existing file: [{"path":"/content/adapter.safetensors","strength":1.0}]. Empty [] is the default; no speed LoRA is added.
import json
from pathlib import Path
# BEGIN EMBEDDED FAST H3 TOOLKIT
FAST_H3_SOURCES = {'fast_h3_runtime.py': '"""L4-only FastH3 runtime shared by the notebook and agent ComfyUI flow.\n\nThe host imports only the standard library. Model dependencies are isolated in\na Colab venv, and the server listens on loopback. No original H3 is downloaded.\n"""\nfrom __future__ import annotations\n\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom urllib.request import urlopen\nimport uuid\nimport zipfile\n\nfrom fast_h3_client import ComfyClient, GraphValidationError, PromptExecutionError, PromptTimeoutError\nfrom fast_h3_workflow import build_graph, to_ui_workflow\n\n\ndef write_json(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_media(path, *, width, height, duration_seconds):\n    from fractions import Fraction\n    command = ["ffprobe", "-v", "error", "-count_frames", "-show_streams", "-show_format", "-of", "json", str(path)]\n    result = subprocess.run(command, capture_output=True, text=True, check=True)\n    info = json.loads(result.stdout)\n    video = next(s for s in info["streams"] if s["codec_type"] == "video")\n    audio = next(s for s in info["streams"] if s["codec_type"] == "audio")\n    if (video["width"], video["height"]) != (width, height):\n        raise ValueError("Decoded video dimensions differ from the requested graph")\n    if float(Fraction(video["avg_frame_rate"])) != 24 or int(video["nb_read_frames"]) != round(duration_seconds * 24):\n        raise ValueError("Decoded video frame count/rate differs from the requested graph")\n    if abs(float(video["duration"]) - duration_seconds) > 1 / 24:\n        raise ValueError("Video duration differs from the requested graph")\n    if int(audio["sample_rate"]) != 32000 or int(audio["channels"]) != 2:\n        raise ValueError("Output must have native 32 kHz stereo audio")\n    if abs(float(audio["duration"]) - duration_seconds) > 0.06:\n        raise ValueError("Audio duration differs from the requested graph")\n    subprocess.run(["ffmpeg", "-v", "error", "-xerror", "-i", str(path), "-f", "null", "-"],\n                   capture_output=True, check=True)\n    return info\n\n\nclass ResourceMonitor:\n    """Observe the notebook and its Comfy subprocess tree, not just notebook RSS."""\n    def __init__(self, state_dir):\n        self.state_dir = Path(state_dir)\n        self.stop_event = threading.Event()\n        self.thread = None\n\n    def sample(self):\n        import psutil\n        vm = psutil.virtual_memory()\n        parent = psutil.Process()\n        processes = [parent] + parent.children(recursive=True)\n        process_info = []\n        for process in processes:\n            try:\n                process_info.append({"pid": process.pid, "rss_bytes": process.memory_info().rss,\n                                     "alive": process.is_running()})\n            except psutil.Error:\n                pass\n        gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv,noheader,nounits"],\n                             capture_output=True, text=True, timeout=15)\n        result = {"time_utc": datetime.now(timezone.utc).isoformat(),\n                  "ram_available_bytes": vm.available, "ram_total_bytes": vm.total,\n                  "processes": process_info, "tree_rss_bytes": sum(p["rss_bytes"] for p in process_info),\n                  "disk_free_bytes": shutil.disk_usage(self.state_dir).free,\n                  "gpu": gpu.stdout.strip() if gpu.returncode == 0 else "unavailable"}\n        cgroup = Path("/sys/fs/cgroup")\n        for key in ("memory.current", "memory.max", "memory.events"):\n            try:\n                result[key] = (cgroup / key).read_text().strip()\n            except OSError:\n                result[key] = "unavailable"\n        for name in ("install.log", "server.log"):\n            file = self.state_dir / name\n            result[name + "_bytes"] = file.stat().st_size if file.exists() else 0\n        cache = self.state_dir.parent / "model-cache"\n        result["download_bytes"] = sum(p.stat().st_size for p in cache.rglob("*") if p.is_file()) if cache.exists() else 0\n        write_json(self.state_dir / "telemetry.json", result)\n        with (self.state_dir / "resources.jsonl").open("a") as stream:\n            stream.write(json.dumps(result) + "\\n")\n\n    def start(self):\n        if self.thread and self.thread.is_alive():\n            return\n        self.stop_event.clear()\n        def loop():\n            while not self.stop_event.is_set():\n                try:\n                    self.sample()\n                except Exception as error:\n                    write_json(self.state_dir / "monitor-error.json", {"error_type": type(error).__name__})\n                self.stop_event.wait(5)\n        self.thread = threading.Thread(target=loop, daemon=True)\n        self.thread.start()\n\n    def stop(self):\n        self.stop_event.set()\n        if self.thread:\n            self.thread.join(timeout=20)\n\n\nclass FastH3Runtime:\n    def __init__(self, root="/content/fast-h3", port=8188):\n        self.root = Path(root)\n        self.state = self.root / "state"\n        self.comfy = self.root / "ComfyUI"\n        self.venv = self.root / "venv"\n        self.python = self.venv / "bin/python"\n        self.port = int(port)\n        self.state.mkdir(parents=True, exist_ok=True)\n        self.lock = json.loads(Path(__file__).with_name("fast_h3_lock.json").read_text())\n        self.client = ComfyClient(f"http://127.0.0.1:{self.port}")\n        self.monitor = ResourceMonitor(self.state)\n        self.server = None\n\n    def phase(self, phase, **extra):\n        write_json(self.state / "progress.json", {"phase": phase, "updated_utc": datetime.now(timezone.utc).isoformat(), **extra})\n        print("FastH3:", phase, flush=True)\n\n    def check_hardware(self):\n        import psutil\n        result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader,nounits"],\n                                capture_output=True, text=True, check=True)\n        rows = [row.split(",") for row in result.stdout.strip().splitlines()]\n        if len(rows) != 1 or rows[0][0].strip() != "NVIDIA L4":\n            raise RuntimeError("This FastH3 test flow requires exactly one NVIDIA L4; no A100/base-model fallback")\n        row = [x.strip() for x in rows[0]]\n        vm = psutil.virtual_memory()\n        hardware = {"gpu": row[0], "vram_total_mib": float(row[1]), "vram_free_mib": float(row[2]),\n                    "driver": row[3], "ram_total_gib": round(vm.total / 2**30, 2),\n                    "ram_available_gib": round(vm.available / 2**30, 2),\n                    "disk_free_gib": round(shutil.disk_usage(self.root).free / 2**30, 2),\n                    "python": sys.version.split()[0]}\n        write_json(self.state / "hardware.json", hardware)\n        return hardware\n\n    def _run(self, command, *, cwd=None):\n        with (self.state / "install.log").open("a") as log:\n            result = subprocess.run([str(x) for x in command], cwd=cwd, stdout=log, stderr=subprocess.STDOUT)\n        if result.returncode:\n            raise RuntimeError(f"{Path(str(command[0])).name} exited {result.returncode}; see {self.state / \'install.log\'}")\n\n    def _checkout(self, repo, revision, destination):\n        destination = Path(destination)\n        if (destination / ".git").is_dir():\n            actual = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip()\n            if actual != revision:\n                raise RuntimeError(f"Existing checkout has a different revision: {destination}")\n            return\n        if destination.exists() and any(destination.iterdir()):\n            raise RuntimeError(f"Refusing to replace a nonempty installation: {destination}")\n        destination.mkdir(parents=True, exist_ok=True)\n        self._run(["git", "init", str(destination)])\n        self._run(["git", "-C", destination, "remote", "add", "origin", repo])\n        self._run(["git", "-C", destination, "fetch", "--depth", "1", "origin", revision])\n        self._run(["git", "-C", destination, "checkout", "--detach", "FETCH_HEAD"])\n\n    def _install(self):\n        self.phase("installing")\n        self._checkout(**dict(self.lock["comfyui"], destination=self.comfy))\n        pack = self.comfy / "custom_nodes/ComfyUI-FastH3-VSA"\n        self._checkout(**dict(self.lock["vsa_pack"], destination=pack))\n        signature = hashlib.sha256(json.dumps({key: self.lock[key] for key in\n                                   ("comfyui", "vsa_pack", "comfy_kitchen")}, sort_keys=True).encode()).hexdigest()\n        marker = self.state / "installed.json"\n        if not marker.exists() or json.loads(marker.read_text()).get("signature") != signature:\n            # Colab can disable ensurepip while exposing its existing pip module.\n            # This also repairs a venv left incomplete by ensurepip failure.\n            self._run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", str(self.venv)])\n            self._run([self.python, "-m", "pip", "--version"])\n            import importlib.metadata\n            constraints = []\n            for package in ("torch", "torchvision", "torchaudio"):\n                try:\n                    constraints.append(f"{package}=={importlib.metadata.version(package)}")\n                except importlib.metadata.PackageNotFoundError:\n                    pass\n            constraint_file = self.state / "torch-constraints.txt"\n            constraint_file.write_text("\\n".join(constraints) + "\\n")\n            self._run([self.python, "-m", "pip", "install", "--disable-pip-version-check", "-c", constraint_file,\n                       "-r", self.comfy / "requirements.txt", "huggingface_hub==1.11.0"])\n            self._run([self.python, "-m", "pip", "freeze"])\n            versions = subprocess.check_output([self.python, "-m", "pip", "freeze"], text=True)\n            (self.state / "versions.txt").write_text(versions)\n            write_json(marker, {"signature": signature, "lock": self.lock})\n        own_package = self.comfy / "custom_nodes/FastH3-L4-Flow"\n        own_package.mkdir(exist_ok=True)\n        shutil.copyfile(Path(__file__).with_name("fast_h3_nodes.py"), own_package / "__init__.py")\n\n    def _kernel_preflight(self):\n        self.phase("kernel_preflight")\n        script = \'\'\'import json, torch, comfy_kitchen\nfrom comfy_kitchen.backends import cuda\nassert torch.cuda.get_device_capability() == (8, 9)\nassert cuda._EXT_AVAILABLE, str(cuda._EXT_ERROR)\nfor name in ("sol_attn_plan", "sol_producer_begin", "sol_producer_chunk", "sol_attn_core"):\n    assert callable(getattr(cuda._C, name, None)), name\nq = torch.randn(1, 256, 4, 128, device="cuda", dtype=torch.bfloat16)\nk = torch.randn_like(q); v = torch.randn_like(q)\nout = cuda.sol_attn(q, k, v, topk_ratio=0.1, tail=False,\n                   sink_blocks=[0,1], sink_q=[0,1],\n                   block_len=torch.full((4,),64,device="cuda",dtype=torch.int32),\n                   coarse_gate=torch.zeros_like(q))\nassert out.shape == q.shape and torch.isfinite(out).all()\ntorch.cuda.synchronize()\nprint(json.dumps({"torch":torch.__version__, "cuda":torch.version.cuda,\n                  "comfy_kitchen":comfy_kitchen.__version__ if hasattr(comfy_kitchen,"__version__") else "unknown",\n                  "device":torch.cuda.get_device_name(), "vsa_kernel":"sol_attn_cuda", "status":"pass"}))\n\'\'\'\n        result = subprocess.run([str(self.python), "-c", script], capture_output=True, text=True)\n        (self.state / "kernel-preflight.log").write_text(result.stdout + result.stderr)\n        if result.returncode:\n            raise RuntimeError("L4 native Sol-Attn kernel preflight failed; model downloads have not started")\n        write_json(self.state / "kernel-preflight.json", json.loads(result.stdout.strip().splitlines()[-1]))\n\n    def _lora_assets(self, loras):\n        if not isinstance(loras, list):\n            raise ValueError("loras must be a JSON list")\n        assets, resolved = [], []\n        for item in loras:\n            if not isinstance(item, dict):\n                raise ValueError("Each LoRA must be an object")\n            strength = item.get("strength", item.get("strength_model", 1.0))\n            if type(strength) not in (int, float) or not math.isfinite(strength) or not -2 <= strength <= 2:\n                raise ValueError("LoRA strength must be a finite value in [-2,2]")\n            if "path" in item:\n                path = Path(item["path"])\n                if not path.is_file() or path.suffix != ".safetensors":\n                    raise ValueError("Local LoRA path must be an existing safetensors file")\n                digest = sha256(path)\n                name = digest[:12] + "_" + path.name\n                assets.append({"local_path": str(path), "size": path.stat().st_size,\n                               "sha256": digest, "destination": "loras/" + name})\n            else:\n                repo, revision, file = item.get("repo", ""), item.get("revision", ""), item.get("file", "")\n                if (not re.fullmatch(r"[\\w.-]+/[\\w.-]+", repo) or not re.fullmatch(r"[0-9a-f]{40}", revision)\n                        or not file.endswith(".safetensors") or file.startswith("/") or ".." in Path(file).parts):\n                    raise ValueError("Remote LoRA requires repo, exact 40-character revision, and safetensors file")\n                with urlopen(f"https://huggingface.co/api/models/{repo}/revision/{revision}?blobs=true", timeout=30) as response:\n                    info = json.load(response)\n                sibling = next((x for x in info["siblings"] if x["rfilename"] == file), None)\n                if sibling is None or not sibling.get("lfs", {}).get("sha256"):\n                    raise ValueError("Pinned LoRA file has no verifiable HF LFS hash")\n                digest = sibling["lfs"]["sha256"]\n                name = digest[:12] + "_" + Path(file).name\n                assets.append({"repo": repo, "revision": revision, "file": file,\n                               "sha256": digest, "size": sibling["size"], "destination": "loras/" + name})\n            resolved.append({"lora_name": name, "strength_model": float(strength)})\n        return assets, resolved\n\n    def _download(self, extra_assets):\n        self.phase("downloading")\n        assets = self.lock["assets"] + extra_assets\n        verified_file = self.state / "verified-assets.json"\n        verified = json.loads(verified_file.read_text()) if verified_file.exists() else {}\n        missing = []\n        for item in assets:\n            path = self.comfy / "models" / item["destination"]\n            if path.exists() and path.stat().st_size == item["size"]:\n                record = verified.get(item["destination"], {})\n                if record.get("sha256") == item["sha256"] and record.get("mtime_ns") == path.stat().st_mtime_ns:\n                    continue\n                if sha256(path) == item["sha256"]:\n                    verified[item["destination"]] = {"sha256": item["sha256"], "mtime_ns": path.stat().st_mtime_ns}\n                    continue\n                raise ValueError("Existing model file has a different SHA256: " + item["destination"])\n            missing.append(item)\n        missing_bytes = sum(item["size"] for item in missing)\n        if shutil.disk_usage(self.root).free < missing_bytes + 8 * 2**30:\n            raise RuntimeError("Insufficient disk for selected FastH3 files and output workspace")\n        write_json(self.state / "model-manifest.json", assets)\n        write_json(self.state / "download-job.json", {"root": str(self.root), "assets": missing})\n        # HF dependencies belong to the isolated environment, not the notebook.\n        download_script = \'\'\'import hashlib,json,os,shutil\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nos.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"]="0"\nos.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"]="4"\nfrom huggingface_hub import hf_hub_download\nfrom huggingface_hub.utils import disable_progress_bars\ndisable_progress_bars()\njob=json.loads(Path(__import__("sys").argv[1]).read_text());root=Path(job["root"])\ndef one(item):\n dst=root/"ComfyUI/models"/item["destination"];dst.parent.mkdir(parents=True,exist_ok=True)\n if dst.exists():raise RuntimeError("Partial destination exists: "+item["destination"])\n if "local_path" in item:src=Path(item["local_path"])\n else:\n  cache=root/"model-cache"/item["repo"].replace("/","--")/item["revision"]\n  src=Path(hf_hub_download(item["repo"],item["file"],revision=item["revision"],local_dir=cache,token=False))\n digest=hashlib.sha256()\n with src.open("rb") as f:\n  for chunk in iter(lambda:f.read(8*1024*1024),b""):digest.update(chunk)\n assert src.stat().st_size==item["size"] and digest.hexdigest()==item["sha256"],item["destination"]\n if "local_path" in item:shutil.copyfile(src,dst)\n else:os.link(src,dst)\n print("VERIFIED",item["destination"],flush=True)\nwith ThreadPoolExecutor(max_workers=2) as pool:list(pool.map(one,job["assets"]))\n\'\'\'\n        self._run([self.python, "-c", download_script, self.state / "download-job.json"])\n        for item in missing:\n            path = self.comfy / "models" / item["destination"]\n            verified[item["destination"]] = {"sha256": item["sha256"], "mtime_ns": path.stat().st_mtime_ns}\n        write_json(verified_file, verified)\n\n    def _server_ready(self):\n        try:\n            self.client.get_json("/system_stats")\n            marker = self.state / "server.json"\n            if not marker.exists():\n                raise RuntimeError("The selected local port belongs to another server")\n            return True\n        except (OSError, ConnectionError):\n            return False\n\n    def _start_server(self):\n        if self._server_ready():\n            return\n        self.phase("starting_server")\n        environment = dict(os.environ, FASTH3_STATE_DIR=str(self.state))\n        log = (self.state / "server.log").open("a")\n        command = [str(self.python), str(self.comfy / "main.py"), "--listen", "127.0.0.1", "--port", str(self.port),\n                   "--disable-auto-launch", "--lowvram", "--disable-pinned-memory", "--cache-lru", "1",\n                   "--reserve-vram", "3", "--use-pytorch-cross-attention", "--enable-triton-backend",\n                   "--disable-comfy-compiler"]\n        self.server = subprocess.Popen(command, cwd=self.comfy, env=environment, stdout=log, stderr=subprocess.STDOUT)\n        log.close()\n        write_json(self.state / "server.json", {"pid": self.server.pid, "port": self.port, "command": command})\n        deadline = time.monotonic() + 180\n        while time.monotonic() < deadline:\n            if self.server.poll() is not None:\n                raise RuntimeError("ComfyUI startup failed; see server.log")\n            if self._server_ready():\n                required = {"FastH3VSAPatch", "FastH3ExactSigmas", "FastH3CheckedLoraLoader", "FastH3ReportAndTrim"}\n                if not required <= self.client.object_info().keys():\n                    raise RuntimeError("Required public/local FastH3 nodes failed to import")\n                return\n            time.sleep(2)\n        raise TimeoutError("ComfyUI did not start within 180 seconds")\n\n    def setup(self, loras=None):\n        started = time.monotonic()\n        hardware = self.check_hardware()\n        extra_assets, resolved = self._lora_assets([] if loras is None else loras)\n        self.monitor.start()\n        try:\n            self._install()\n            self._kernel_preflight()\n            self._download(extra_assets)\n            self._start_server()\n            result = {"status": "ready", "hardware": hardware, "lock": self.lock,\n                      "loras": resolved, "setup_seconds": round(time.monotonic() - started, 2)}\n            write_json(self.state / "setup.json", result)\n            self.phase("ready")\n            return result\n        except Exception as error:\n            self.phase("setup_failed", error_type=type(error).__name__)\n            raise\n        finally:\n            self.monitor.stop()\n\n    def ensure_ready(self, loras=None):\n        assets, resolved = self._lora_assets([] if loras is None else loras)\n        if not self._server_ready():\n            return self.setup(loras=loras)\n        self.check_hardware()\n        if assets:\n            self.monitor.start()\n            try:\n                self._download(assets)\n            finally:\n                self.monitor.stop()\n        return resolved\n\n    def status(self):\n        result = {"root": str(self.root), "server_ready": self._server_ready()}\n        for name in ("progress.json", "hardware.json", "kernel-preflight.json"):\n            file = self.state / name\n            if file.exists():\n                result[name[:-5]] = json.loads(file.read_text())\n        if result["server_ready"]:\n            result["queue"] = self.client.get_json("/queue")\n        return result\n\n    def _reference_inputs(self, reference_paths, *, stage=True):\n        if not isinstance(reference_paths, list) or not 1 <= len(reference_paths) <= 3:\n            raise ValueError("Experimental Ref2VA requires 1–3 local reference image paths")\n        manifest, names = [], []\n        for index, value in enumerate(reference_paths):\n            if not isinstance(value, str):\n                raise ValueError("Reference paths must be strings")\n            source = Path(value).expanduser().resolve(strict=True)\n            if not source.is_file() or source.suffix.lower() not in (".png", ".jpg", ".jpeg", ".webp"):\n                raise ValueError("Reference must be a local PNG/JPEG/WebP file")\n            from PIL import Image\n            with Image.open(source) as img:\n                img.load()\n                size = list(img.size)\n            digest = sha256(source)\n            # Stock LoadImage advertises only files directly in its input root.\n            name = "fast_h3_ref_" + digest + source.suffix.lower()\n            if name in names:\n                raise ValueError("Do not repeat identical reference files")\n            target = self.comfy / "input" / name\n            if stage:\n                target.parent.mkdir(parents=True, exist_ok=True)\n                if not target.exists():\n                    shutil.copyfile(source, target)\n                if sha256(target) != digest:\n                    raise ValueError("Reference input copy failed SHA256 verification")\n            names.append(name)\n            manifest.append({"picture": index + 1, "input_name": name, "sha256": digest,\n                             "bytes": source.stat().st_size, "source_size": size})\n        return names, manifest\n\n    def generate(self, prompt, width=864, height=480, duration_seconds=5, seed=123456662,\n                 loras=None, run_name="fast_h3_notebook", timeout=1800,\n                 generation_mode="t2va", reference_paths=None):\n        # Validate settings before any setup/download or queue side effect.\n        reference_paths = [] if reference_paths is None else reference_paths\n        if not isinstance(reference_paths, list):\n            raise ValueError("reference_paths must be a JSON list")\n        build_graph(prompt, width, height, duration_seconds, seed, run_name=run_name,\n                    generation_mode=generation_mode,\n                    reference_images=[f"reference_{i}.png" for i in range(len(reference_paths))])\n        names, reference_manifest = self._reference_inputs(reference_paths, stage=False) if reference_paths else ([], [])\n        _, resolved = self._lora_assets([] if loras is None else loras)\n        self.ensure_ready(loras=loras)\n        if reference_paths:\n            names, reference_manifest = self._reference_inputs(reference_paths)\n        identity = str(uuid.uuid4())\n        run_dir = self.root / "runs" / (run_name + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + identity[:8])\n        run_dir.mkdir(parents=True, exist_ok=False)\n        graph = build_graph(prompt, width, height, duration_seconds, seed, loras=resolved,\n                            run_name=run_dir.name, generation_mode=generation_mode, reference_images=names,\n                            request_id=identity)\n        workflow = to_ui_workflow(graph)\n        write_json(run_dir / "workflow.api.json", graph)\n        write_json(run_dir / "workflow.json", workflow)\n        if reference_manifest:\n            write_json(run_dir / "reference-manifest.json", reference_manifest)\n        metrics = {"status": "submitting", "prompt_id": identity, "run_dir": str(run_dir),\n                   "width": width, "height": height, "duration_seconds": duration_seconds,\n                   "prompt": prompt, "seed": seed, "lock": self.lock, "loras": resolved,\n                   "generation_mode": generation_mode, "reference_manifest": reference_manifest}\n        started = time.monotonic()\n        write_json(run_dir / "metrics.json", metrics)\n        with (run_dir / "started.json").open("x") as stream:\n            json.dump({"prompt_id": identity}, stream)\n        self.monitor.start()\n        try:\n            self.phase("submitting", prompt_id=identity, run_dir=str(run_dir))\n            response = self.client.queue_prompt(graph, prompt_id=identity,\n                        extra_data={"extra_pnginfo": {"workflow": workflow}})\n            if response["prompt_id"] != identity:\n                raise RuntimeError("ComfyUI did not retain the reserved prompt identity")\n            metrics["status"] = "queued"\n            write_json(run_dir / "metrics.json", metrics)\n            history = self.client.wait_prompt(identity, timeout=timeout, on_status=lambda data:\n                        self.phase(data["state"], prompt_id=identity, run_dir=str(run_dir), elapsed_seconds=data["elapsed_seconds"]))\n            write_json(run_dir / "history.json", history)\n            reports = [report for output in history.get("outputs", {}).values() for report in output.get("fast_h3", [])]\n            if len(reports) != 1:\n                raise RuntimeError("ComfyUI completed without the FastH3 execution receipt")\n            if generation_mode == "experimental_ref2va_ood":\n                report = reports[0]\n                reference = report.get("reference", {})\n                if (report.get("mode") != generation_mode or reference.get("reference_count") != len(names)\n                        or len(reference.get("forward_seconds", [])) != 4\n                        or len(reference.get("layouts", [])) != 4\n                        or any(item.get("ref_image_tokens", 0) <= 0 for item in reference.get("layouts", []))):\n                    raise RuntimeError("Ref2VA receipt did not prove four image-conditioned forwards")\n            outputs = self.client.output_files(history)\n            if len(outputs) != 1:\n                raise RuntimeError("Expected one saved FastH3 MP4")\n            self.phase("verifying", prompt_id=identity, run_dir=str(run_dir))\n            video = self.client.download_output(outputs[0], run_dir / "video.mp4")\n            media = verify_media(video, width=width, height=height, duration_seconds=duration_seconds)\n            write_json(run_dir / "media.json", media)\n            metrics.update(reports[0], status="technically_valid", video_path=str(video),\n                           video_sha256=sha256(video), total_seconds=round(time.monotonic() - started, 2))\n            for name in ("hardware.json", "kernel-preflight.json", "versions.txt", "model-manifest.json", "resources.jsonl"):\n                file = self.state / name\n                if file.exists():\n                    shutil.copyfile(file, run_dir / name)\n            write_json(run_dir / "metrics.json", metrics)\n            archive = run_dir.with_suffix(".zip")\n            with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as bundle:\n                for file in run_dir.iterdir():\n                    if file.is_file():\n                        bundle.write(file, file.name)\n            metrics["archive_path"] = str(archive)\n            write_json(self.root / "latest.json", metrics)\n            self.phase("technically_valid", prompt_id=identity, run_dir=str(run_dir))\n            print("PASS: FastH3, four Euler forwards, 200 native VSA calls, video + stereo audio", flush=True)\n            return metrics\n        except Exception as error:\n            if isinstance(error, GraphValidationError):\n                metrics["status"] = "validation_failed"\n            elif isinstance(error, PromptExecutionError):\n                write_json(run_dir / "history.json", error.history)\n                metrics["status"] = "execution_failed"\n            else:\n                metrics["status"] = "needs_reconcile"\n            metrics["error_type"] = type(error).__name__\n            metrics["total_seconds"] = round(time.monotonic() - started, 2)\n            write_json(run_dir / "metrics.json", metrics)\n            self.phase(metrics["status"], prompt_id=identity, run_dir=str(run_dir))\n            raise\n        finally:\n            self.monitor.stop()\n\n    def stop_server(self):\n        marker = self.state / "server.json"\n        if not marker.exists():\n            return\n        import psutil\n        pid = json.loads(marker.read_text())["pid"]\n        try:\n            process = psutil.Process(pid)\n            if str(self.comfy / "main.py") not in process.cmdline():\n                raise RuntimeError("Server PID now belongs to another process")\n            process.terminate()\n            process.wait(timeout=20)\n        except psutil.NoSuchProcess:\n            pass\n        self.phase("server_stopped")\n', 'fast_h3_workflow.py': '"""Recreated FastH3 text-to-video/audio graphs; no GPU imports or downloads."""\n\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport re\n\n\nDEFAULT_PROMPT = (\n    "integrated_multimodal_description: [Shot 1] A small red toy train rolls slowly "\n    "from left to right along a wooden track on a clean tabletop. The train stays "\n    "fully visible in a steady wide shot; its wheels turn as it moves. No dialogue.\\n"\n    "overall_soundscape: Quiet room ambience and gentle wheel clicks on the wooden track.\\n"\n    "non_diegetic_music: N/A"\n)\nUNET_NAME = "minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors"\nCLIP_NAME = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"\nVIDEO_VAE_NAME = "minimax_h3_video_vae_int8_convrot.safetensors"\nAUDIO_VAE_NAME = "minimax_h3_audio_vae_fp32.safetensors"\n\n\ndef model_frame_count(duration_seconds):\n    """Round 24 fps output up to the native 17k+5 grid before final trimming."""\n    if type(duration_seconds) is not int or not 5 <= duration_seconds <= 15:\n        raise ValueError("duration_seconds must be an integer from 5 to 15")\n    return ((duration_seconds * 24 - 5 + 16) // 17) * 17 + 5\n\n\ndef build_graph(prompt, width=864, height=480, duration_seconds=5,\n                seed=123456662, loras=None, run_name="fast_h3",\n                generation_mode="t2va", reference_images=None, request_id=None):\n    length = model_frame_count(duration_seconds)\n    if any(type(value) is not int or value <= 0 or value % 32 for value in (width, height)):\n        raise ValueError("width and height must be positive integer multiples of 32")\n    if width * height > 1344 * 768:\n        raise ValueError("requested area exceeds 1344 x 768")\n    if type(seed) is not int or not 0 <= seed <= 2**64 - 1:\n        raise ValueError("seed must be an integer from 0 to 2^64-1")\n    if generation_mode not in ("t2va", "experimental_ref2va_ood"):\n        raise ValueError("generation_mode must be t2va or experimental_ref2va_ood")\n    reference_images = [] if reference_images is None else reference_images\n    if not isinstance(reference_images, list):\n        raise ValueError("reference_images must be a resolved list of ComfyUI input filenames")\n    experimental = generation_mode == "experimental_ref2va_ood"\n    if (experimental and not 1 <= len(reference_images) <= 3) or (not experimental and reference_images):\n        raise ValueError("Experimental Ref2VA requires 1–3 images; T2VA requires none")\n    for name in reference_images:\n        if (not isinstance(name, str) or not name or "\\\\" in name or ".." in PurePosixPath(name).parts\n                or PurePosixPath(name).is_absolute() or PurePosixPath(name).suffix.lower() not in (".png", ".jpg", ".jpeg", ".webp")):\n            raise ValueError("References must be relative ComfyUI input image filenames")\n    if len(set(reference_images)) != len(reference_images):\n        raise ValueError("Do not repeat the same reference file")\n    fields = (("subject_definitions", "summary", "retention_analysis", "detailed_description", "overall_soundscape", "non_diegetic_music")\n              if experimental else ("integrated_multimodal_description", "overall_soundscape", "non_diegetic_music"))\n    pattern = r"\\A\\s*" + r"\\s*".join(re.escape(field) + r":\\s*(.+?)" for field in fields) + r"\\s*\\Z"\n    match = re.fullmatch(pattern, prompt, flags=re.DOTALL) if isinstance(prompt, str) else None\n    if match is None or not all(value.strip() for value in match.groups()):\n        raise ValueError("prompt must contain all nonempty H3 mode fields in canonical order")\n    tags = re.findall(r"<(Picture|Video|Audio)\\s+(\\d+)>", prompt, flags=re.IGNORECASE)\n    if not experimental and tags:\n        raise ValueError("reference tags are unsupported by this text-only workflow")\n    if experimental:\n        if any(kind.lower() != "picture" for kind, _ in tags):\n            raise ValueError("This experiment supports image references only")\n        if {int(number) for _, number in tags} != set(range(1, len(reference_images) + 1)):\n            raise ValueError("Picture tags must match the ordered reference images exactly")\n        if not match.group(2).lstrip().startswith("[reference generation]"):\n            raise ValueError("Ref2VA summary must start with [reference generation]")\n    if not isinstance(run_name, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", run_name):\n        raise ValueError("run_name must be a filename-safe name without path separators")\n    if loras is None:\n        loras = []\n    if not isinstance(loras, list):\n        raise ValueError("loras must be a resolved list of lora_name/strength_model dictionaries")\n\n    def node(class_type, **inputs):\n        return {"class_type": class_type, "inputs": inputs}\n\n    graph = {"1": node("UNETLoader", unet_name=UNET_NAME, weight_dtype="default")}\n    model = ["1", 0]\n    for index, lora in enumerate(loras):\n        if not isinstance(lora, dict) or set(lora) != {"lora_name", "strength_model"}:\n            raise ValueError("each LoRA must have exactly lora_name and strength_model")\n        name, strength = lora["lora_name"], lora["strength_model"]\n        if (not isinstance(name, str) or not name or "\\\\" in name\n                or PurePosixPath(name).is_absolute() or ".." in PurePosixPath(name).parts\n                or not name.endswith(".safetensors")):\n            raise ValueError("lora_name must be a relative .safetensors model filename")\n        if type(strength) not in (int, float) or not math.isfinite(strength):\n            raise ValueError("strength_model must be a finite number")\n        identity = str(100 + index)\n        graph[identity] = node("FastH3CheckedLoraLoader", model=model, lora_name=name,\n                               strength_model=float(strength))\n        model = [identity, 0]\n    graph.update({\n        "2": node("MiniMaxH3SigmaShift", model=model, shift_video=12.0, shift_audio=3.0),\n        "3": node("FastH3VSAPatch", model=["2", 0], keep_percent=10.0, min_tokens=8192,\n                  failure_mode="error", allow_unsupported_conditioning=experimental, verbose=True),\n        "4": node("CLIPLoader", clip_name=CLIP_NAME, type="minimax", device="default"),\n        "5": node("VAELoader", vae_name=VIDEO_VAE_NAME),\n        "6": node("VAELoader", vae_name=AUDIO_VAE_NAME),\n        "7": node("MiniMaxH3ImageToVideo", clip=["4", 0], vae=["5", 0], prompt=prompt,\n                  width=width, height=height, length=length),\n        "8": node("BasicGuider", model=["3", 0], conditioning=["7", 0]),\n        "9": node("FastH3ExactSigmas", model=["3", 0]),\n        "10": node("KSamplerSelect", sampler_name="euler"),\n        "11": node("RandomNoise", noise_seed=seed),\n        "12": node("SamplerCustomAdvanced", noise=["11", 0], guider=["8", 0], sampler=["10", 0],\n                   sigmas=["9", 0], latent_image=["7", 1]),\n        "13": node("VAEDecode", samples=["12", 0], vae=["5", 0]),\n        "14": node("VAEDecodeAudio", samples=["12", 0], vae=["6", 0]),\n        "15": node("FastH3ReportAndTrim", model=["3", 0], sigmas=["9", 0], images=["13", 0],\n                   audio=["14", 0], duration_seconds=float(duration_seconds)),\n        "16": node("CreateVideo", images=["15", 0], audio=["15", 1], fps=24.0),\n        "17": node("SaveVideo", video=["16", 0], filename_prefix="fast_h3/" + run_name,\n                   **{"format": "mp4", "format.codec": "h264", "format.codec.encoding": "auto"}),\n    })\n    if experimental:\n        for index, name in enumerate(reference_images):\n            graph[str(200 + index)] = node("LoadImage", image=name)\n        graph["7"] = node("FastH3ExperimentalReference", model=["3", 0], clip=["4", 0], vae=["5", 0],\n                          prompt=prompt, width=width, height=height, length=length,\n                          **{f"reference_{index + 1}": [str(200 + index), 0] for index in range(len(reference_images))})\n        graph["8"]["inputs"].update(model=["7", 0], conditioning=["7", 1])\n        graph["9"]["inputs"]["model"] = ["7", 0]\n        graph["12"]["inputs"]["latent_image"] = ["7", 2]\n        graph["15"]["inputs"]["model"] = ["7", 0]\n        graph["19"] = node("FastH3SampleRequest", noise=["11", 0], request_id=request_id or run_name)\n        graph["12"]["inputs"]["noise"] = ["19", 0]\n    return graph\n\n\n# Input order and widget types follow ComfyUI/VSA node schemas. Seed-control and\n# optional CreateVideo widgets are frontend settings, not required API inputs.\n_SCHEMAS = {\n    "UNETLoader": ([("unet_name", "COMBO"), ("weight_dtype", "COMBO")], ["MODEL"]),\n    "FastH3CheckedLoraLoader": ([("model", "MODEL"), ("lora_name", "COMBO"), ("strength_model", "FLOAT")], ["MODEL"]),\n    "MiniMaxH3SigmaShift": ([("model", "MODEL"), ("shift_video", "FLOAT"), ("shift_audio", "FLOAT")], ["MODEL"]),\n    "FastH3VSAPatch": ([("model", "MODEL"), ("keep_percent", "FLOAT"), ("min_tokens", "INT"),\n                        ("failure_mode", "COMBO"), ("allow_unsupported_conditioning", "BOOLEAN"),\n                        ("verbose", "BOOLEAN")], ["MODEL", "STRING"]),\n    "CLIPLoader": ([("clip_name", "COMBO"), ("type", "COMBO"), ("device", "COMBO")], ["CLIP"]),\n    "VAELoader": ([("vae_name", "COMBO")], ["VAE"]),\n    "LoadImage": ([("image", "COMBO")], ["IMAGE", "MASK"]),\n    "FastH3SampleRequest": ([("noise", "NOISE"), ("request_id", "STRING")], ["NOISE"]),\n    "FastH3ExperimentalReference": ([("model", "MODEL"), ("clip", "CLIP"), ("vae", "VAE"),\n        ("prompt", "STRING"), ("width", "INT"), ("height", "INT"), ("length", "INT"),\n        ("reference_1", "IMAGE"), ("reference_2", "IMAGE"), ("reference_3", "IMAGE")], ["MODEL", "CONDITIONING", "LATENT"]),\n    "MiniMaxH3ImageToVideo": ([("clip", "CLIP"), ("vae", "VAE"), ("prompt", "STRING"),\n                              ("width", "INT"), ("height", "INT"), ("length", "INT")], ["CONDITIONING", "LATENT"]),\n    "BasicGuider": ([("model", "MODEL"), ("conditioning", "CONDITIONING")], ["GUIDER"]),\n    "FastH3ExactSigmas": ([("model", "MODEL")], ["SIGMAS", "STRING"]),\n    "KSamplerSelect": ([("sampler_name", "COMBO")], ["SAMPLER"]),\n    "RandomNoise": ([("noise_seed", "INT")], ["NOISE"]),\n    "SamplerCustomAdvanced": ([("noise", "NOISE"), ("guider", "GUIDER"), ("sampler", "SAMPLER"),\n                               ("sigmas", "SIGMAS"), ("latent_image", "LATENT")], ["LATENT", "LATENT"]),\n    "VAEDecode": ([("samples", "LATENT"), ("vae", "VAE")], ["IMAGE"]),\n    "VAEDecodeAudio": ([("samples", "LATENT"), ("vae", "VAE")], ["AUDIO"]),\n    "FastH3ReportAndTrim": ([("model", "MODEL"), ("sigmas", "SIGMAS"), ("images", "IMAGE"),\n                            ("audio", "AUDIO"), ("duration_seconds", "FLOAT")], ["IMAGE", "AUDIO", "STRING"]),\n    "CreateVideo": ([("images", "IMAGE"), ("fps", "FLOAT"), ("audio", "AUDIO")], ["VIDEO"]),\n    "SaveVideo": ([("video", "VIDEO"), ("filename_prefix", "STRING"), ("format", "COMFY_DYNAMICCOMBO_V3"),\n                   ("format.codec", "COMFY_DYNAMICCOMBO_V3"), ("format.codec.encoding", "COMFY_DYNAMICCOMBO_V3")], ["VIDEO"]),\n}\n_POSITIONS = {1: (0, 0), 2: (720, 0), 3: (720, 190), 4: (0, 180), 5: (0, 370),\n              6: (0, 500), 7: (0, 700), 8: (1100, 0), 9: (1100, 170), 10: (1100, 340),\n              11: (1100, 500), 12: (1470, 550), 13: (1860, 0), 14: (1860, 170),\n              15: (1860, 370), 16: (2250, 0), 17: (2640, 0), 19: (1100, 680),\n              200: (-660, 600), 201: (-660, 950), 202: (-660, 1300)}\n\n\ndef to_ui_workflow(graph):\n    """Convert this builder\'s API graph to ComfyUI workflow schema 0.4."""\n    nodes, links, ordered = [], [], []\n    pending = dict(graph)\n    while pending:\n        ready = [identity for identity, node in pending.items()\n                 if all(not isinstance(value, list) or value[0] in ordered for value in node["inputs"].values())]\n        if not ready:\n            raise ValueError("graph has missing link sources or a cycle")\n        for identity in ready:\n            ordered.append(identity)\n            del pending[identity]\n    for order, identity in enumerate(ordered):\n        api = graph[identity]\n        input_specs, output_types = _SCHEMAS[api["class_type"]]\n        if api["class_type"] == "FastH3ExperimentalReference":\n            input_specs = [(name, kind) for name, kind in input_specs\n                           if name not in ("reference_2", "reference_3") or name in api["inputs"]]\n        if set(api["inputs"]) != {name for name, _ in input_specs}:\n            raise ValueError(f"node {identity} inputs differ from supported workflow schema")\n        inputs, widgets = [], []\n        for name, kind in input_specs:\n            value = api["inputs"][name]\n            entry = {"name": name, "type": kind, "link": None}\n            if isinstance(value, list):\n                source, slot = value\n                source_types = _SCHEMAS[graph[source]["class_type"]][1]\n                if type(slot) is not int or not 0 <= slot < len(source_types) or source_types[slot] != kind:\n                    raise ValueError(f"node {identity} has invalid output slot or link type")\n                link_id = len(links) + 1\n                entry["link"] = link_id\n                links.append([link_id, int(source), slot, int(identity), len(inputs), kind])\n            else:\n                entry["widget"] = {"name": name}\n                widgets.append(value)\n            inputs.append(entry)\n        if api["class_type"] == "RandomNoise":\n            widgets.append("fixed")\n        elif api["class_type"] == "CreateVideo":\n            widgets.extend(["auto", "sRGB", "none"])\n            inputs.extend({"name": name, "type": "COMBO", "link": None, "widget": {"name": name}}\n                          for name in ("bit_depth", "color_space", "codec"))\n        node_id = int(identity)\n        width = 620 if node_id in (1, 4, 5, 6, 7) else 330\n        height = 420 if node_id == 7 else 70 + 24 * max(len(widgets), len(inputs))\n        pos = _POSITIONS.get(node_id, (0, 1220 + (node_id - 100) * 190))\n        nodes.append({"id": node_id, "type": api["class_type"], "pos": list(pos), "size": [width, height],\n                      "flags": {}, "order": order, "mode": 0, "inputs": inputs,\n                      "outputs": [{"name": kind, "type": kind, "links": None, "slot_index": slot}\n                                  for slot, kind in enumerate(output_types)],\n                      "properties": {"Node name for S&R": api["class_type"]}, "widgets_values": widgets})\n    by_id = {node["id"]: node for node in nodes}\n    for link_id, source, slot, *_ in links:\n        output = by_id[source]["outputs"][slot]\n        if output["links"] is None:\n            output["links"] = []\n        output["links"].append(link_id)\n    notes = ("## 自行重建的 FastH3 文生影音工作流\\n\\n"\n             "依公開節點重建，並非受限作者原始 JSON。預設 864×480、24 fps、5 秒；"\n             "原生 124 幀經影音裁切後輸出 120 幀。僅文生影音，不含首尾幀或參考素材。\\n\\n"\n             "student → 可選 LoRA → Sigma Shift → 嚴格 VSA → 四步 Euler → 影音解碼 → 裁切 → MP4。"\n             "不預設掛載 Turbo LoRA。L4 已完成 864×480、5 秒文生影音測試，200 次 VSA、零 fallback。\\n\\n"\n             "Colab L4 請用 --disable-comfy-compiler 啟動 ComfyUI，保留動態 VRAM 與 offload。"\n             "其他尺寸／片長及實際內容 LoRA 的品質需另行測試。")\n    if graph["7"]["class_type"] == "FastH3ExperimentalReference":\n        notes = ("## FastH3 圖片參照實驗（未蒸餾 Ref2VA）\\n\\n"\n                 "圖片經原生 MiniMaxH3ReferenceToVideo 送入 Qwen 與 video VAE；每個 DiT forward 核對 ref_img rows。"\n                 "四步 checkpoint 的訓練範圍仍為 T2VA，本接線屬分布外實驗，人物品質須另驗。"\n                 "VSA override 只對此實驗分支開啟；仍要求 200 VSA、零 fallback，不掛 speed LoRA。")\n    nodes.append({"id": 18, "type": "MarkdownNote", "pos": [720, 850], "size": [1090, 260],\n                  "flags": {}, "order": len(nodes), "mode": 0, "inputs": [], "outputs": [],\n                  "properties": {}, "widgets_values": [notes]})\n    groups = []\n    for title, members, color in (("模型與文字條件（可選 LoRA）", {1, 4, 5, 6, 7} | {int(i) for i in graph if int(i) >= 100}, "#3f789e"),\n                                   ("嚴格 VSA 與四步 Euler", set(range(2, 4)) | set(range(8, 13)), "#6b659b"),\n                                   ("影音解碼、裁切與 MP4", set(range(13, 18)), "#4f8a73")):\n        chosen = [node for node in nodes if node["id"] in members]\n        left = min(node["pos"][0] for node in chosen) - 20\n        top = min(node["pos"][1] for node in chosen) - 60\n        right = max(node["pos"][0] + node["size"][0] for node in chosen) + 20\n        bottom = max(node["pos"][1] + node["size"][1] for node in chosen) + 20\n        groups.append({"title": title, "bounding": [left, top, right - left, bottom - top], "color": color, "font_size": 24, "flags": {}})\n    return {"last_node_id": max(max(by_id), 18), "last_link_id": len(links), "nodes": nodes, "links": links,\n            "groups": groups, "config": {}, "extra": {"ds": {"scale": 0.5, "offset": [60, 100]},\n            "fast_h3_notes": notes},\n            "version": 0.4}\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser(description="Export the public FastH3 UI/API workflow; --help is read-only.")\n    parser.add_argument("--output-dir", type=Path, default=Path(__file__).resolve().parents[1] / "workflows")\n    destination = parser.parse_args().output_dir\n    destination.mkdir(parents=True, exist_ok=True)\n    graph = build_graph(DEFAULT_PROMPT)\n    for name, content in (("fast_h3_l4.api.json", graph), ("fast_h3_l4.json", to_ui_workflow(graph))):\n        (destination / name).write_text(json.dumps(content, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n', 'fast_h3_nodes.py': '"""Local ComfyUI nodes for the article\'s FastH3 schedule, LoRAs and receipts.\n\nLearned VSA itself is supplied by the separately pinned public VSA node pack.\nThis file is installed as a custom-node package by fast_h3_runtime.py.\n"""\n\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\n\nclass ReferenceTrace:\n    def __init__(self, *args, **kwargs):\n        self.data = dict(*args, **kwargs)\n\n    def __getitem__(self, key):\n        return self.data[key]\n\n    def __setitem__(self, key, value):\n        self.data[key] = value\n\n    def __deepcopy__(self, memo):\n        # ModelPatcher copies options into sampler/executors; receipts must see\n        # the trace produced by those executors, just like the public VSA runtime.\n        return self\n\n\nclass FastH3SampleRequest:\n    """Execute new sampling while retaining exact seed and cached conditioning."""\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {"required": {"noise": ("NOISE",), "request_id": ("STRING",)}}\n    RETURN_TYPES = ("NOISE",)\n    FUNCTION = "stamp"\n    CATEGORY = "FastH3/sampling"\n\n    def stamp(self, noise, request_id):\n        if not isinstance(request_id, str) or not request_id:\n            raise ValueError("A new sampling request requires an identity")\n        return (noise,)\n\n\ndef _tensor_digest(tensor):\n    import torch\n    value = tensor.detach().contiguous().view(torch.uint8).cpu()\n    return hashlib.sha256(value.numpy().tobytes()).hexdigest()\n\n\ndef _reference_sample_wrapper(executor, *args, **kwargs):\n    trace = executor.class_obj.model_options["fast_h3_reference"]\n    trace["forward_seconds"] = []\n    trace["layouts"] = []\n    started = time.perf_counter()\n    try:\n        return executor(*args, **kwargs)\n    finally:\n        trace["sampler_seconds"] = time.perf_counter() - started\n\n\ndef _reference_diffusion_wrapper(executor, x, timestep, context, transformer_options=None,\n                                 minimax_payload=None, **kwargs):\n    import torch\n    options = transformer_options if transformer_options is not None else {}\n    trace = options["fast_h3_reference_trace"]\n    layout = (minimax_payload or {}).get("layout")\n    segments = list(getattr(layout, "segments", ()))\n    rows = sum(int(end) - int(start) for start, end, kind in segments if kind == "ref_img")\n    if rows <= 0 or rows != trace["expected_ref_image_tokens"]:\n        raise RuntimeError("Reference latent rows did not reach the H3 diffusion-model input")\n    torch.cuda.synchronize()\n    started = time.perf_counter()\n    result = executor(x, timestep, context, options, minimax_payload=minimax_payload, **kwargs)\n    torch.cuda.synchronize()\n    trace["forward_seconds"].append(time.perf_counter() - started)\n    trace["layouts"].append({"ref_image_tokens": rows,\n                             "segments": [[int(a), int(b), kind] for a, b, kind in segments]})\n    return result\n\n\nclass FastH3ExperimentalReference:\n    """Native full Ref2VA conditioning on the unchanged T2VA student: OOD."""\n\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {"required": {\n            "model": ("MODEL",), "clip": ("CLIP",), "vae": ("VAE",),\n            "prompt": ("STRING", {"multiline": True}),\n            "width": ("INT", {"default": 864, "min": 32, "step": 32}),\n            "height": ("INT", {"default": 480, "min": 32, "step": 32}),\n            "length": ("INT", {"default": 124, "min": 5, "step": 17}),\n            "reference_1": ("IMAGE",),\n        }, "optional": {"reference_2": ("IMAGE",), "reference_3": ("IMAGE",)}}\n\n    RETURN_TYPES = ("MODEL", "CONDITIONING", "LATENT")\n    FUNCTION = "condition"\n    CATEGORY = "FastH3/experimental"\n\n    def condition(self, model, clip, vae, prompt, width, height, length,\n                  reference_1, reference_2=None, reference_3=None):\n        import comfy.patcher_extension\n        from comfy_extras.nodes_minimax_h3 import MiniMaxH3ReferenceToVideo\n        if reference_3 is not None and reference_2 is None:\n            raise ValueError("Reference images must be consecutive")\n        images = [image for image in (reference_1, reference_2, reference_3) if image is not None]\n        if not images or vae is None:\n            raise ValueError("Full Ref2VA requires both an image and video VAE")\n        native = MiniMaxH3ReferenceToVideo.execute(\n            clip=clip, vae=vae, prompt=prompt, width=width, height=height, length=length,\n            ref_image_size="match", ref_images={f"ref_image_{i + 1}": image for i, image in enumerate(images)})\n        conditioning, latent = native.result\n        blocks = conditioning[0][1].get("minimax_refs", [])\n        if len(blocks) != len(images) or any(b.get("kind") != "image" or b.get("latent") is None for b in blocks):\n            raise RuntimeError("Native reference conditioning omitted the image VAE latents")\n        trace = ReferenceTrace({\n            "mode": "experimental_ref2va_ood", "supported_by_distillation": False,\n            "native_conditioner": "MiniMaxH3ReferenceToVideo", "reference_count": len(images),\n            "ref_image_size": "match", "forward_seconds": [], "layouts": [],\n            "sources": [{"picture": i + 1, "rgb_shape": list(image.shape), "rgb_tensor_sha256": _tensor_digest(image)}\n                        for i, image in enumerate(images)],\n            "encoded": [{"picture": i + 1, "latent_shape": list(b["latent"].shape),\n                         "latent_sha256": _tensor_digest(b["latent"])} for i, b in enumerate(blocks)],\n            "expected_ref_image_tokens": sum((b["latent_h"] // 2) * (b["latent_w"] // 2) for b in blocks),\n        })\n        result = model.clone()\n        result.model_options = dict(result.model_options)\n        result.model_options["fast_h3_reference"] = trace\n        options = dict(result.model_options.get("transformer_options", {}))\n        options["fast_h3_reference_trace"] = trace\n        result.model_options["transformer_options"] = options\n        result.add_wrapper_with_key(comfy.patcher_extension.WrappersMP.OUTER_SAMPLE,\n                                    "fast_h3_reference_trace", _reference_sample_wrapper)\n        result.add_wrapper_with_key(comfy.patcher_extension.WrappersMP.DIFFUSION_MODEL,\n                                    "fast_h3_reference_trace", _reference_diffusion_wrapper)\n        return result, conditioning, latent\n\n\nclass FastH3ExactSigmas:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {"required": {"model": ("MODEL",)}}\n\n    RETURN_TYPES = ("SIGMAS", "STRING")\n    RETURN_NAMES = ("sigmas", "schedule")\n    FUNCTION = "schedule"\n    CATEGORY = "FastH3/sampling"\n\n    def schedule(self, model):\n        import torch\n        sampling = model.get_model_object("model_sampling")\n        if (float(sampling.shift), float(sampling.audio_shift)) != (12.0, 3.0):\n            raise ValueError("FastH3 requires MiniMaxH3SigmaShift video=12, audio=3")\n        timesteps = [999, 749, 500, 250]\n        sigmas = sampling.sigma(torch.tensor(timesteps, dtype=torch.float32))\n        sigmas = torch.cat((sigmas, sigmas.new_zeros(1)))\n        report = {"timesteps": timesteps, "sigmas": sigmas.tolist(),\n                  "video_shift": 12.0, "audio_shift": 3.0, "forwards": 4}\n        return sigmas, json.dumps(report)\n\n\nclass FastH3CheckedLoraLoader:\n    @classmethod\n    def INPUT_TYPES(cls):\n        import folder_paths\n        return {"required": {\n            "model": ("MODEL",),\n            "lora_name": (folder_paths.get_filename_list("loras"),),\n            "strength_model": ("FLOAT", {"default": 1.0, "min": -2.0, "max": 2.0, "step": 0.05}),\n        }}\n\n    RETURN_TYPES = ("MODEL",)\n    FUNCTION = "load"\n    CATEGORY = "FastH3/LoRA"\n\n    def load(self, model, lora_name, strength_model):\n        import comfy.lora\n        import comfy.utils\n        import folder_paths\n        # FastH3 already contains the speed distillation. These known base-model\n        # speed adapters do not establish compatibility with this student.\n        name = lora_name.lower()\n        if any(word in name for word in ("turbo", "flashgen", "pdmd", "dmad")):\n            raise ValueError("Use a FastH3-compatible content/style LoRA; do not stack a base H3 speed adapter")\n        result = model.clone()\n        path = Path(folder_paths.get_full_path_or_raise("loras", lora_name))\n        digest = hashlib.sha256()\n        with path.open("rb") as stream:\n            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n                digest.update(chunk)\n        matched = 0\n        if strength_model != 0:\n            state = comfy.utils.load_torch_file(str(path), safe_load=True)\n            keys = comfy.lora.model_lora_keys_unet(model.model, {})\n            patches = comfy.lora.load_lora(state, keys)\n            if not patches:\n                raise ValueError("LoRA has no targets recognized by the FastH3 ComfyUI model")\n            applied = result.add_patches(patches, strength_model)\n            if len(applied) != len(patches):\n                raise ValueError(f"LoRA targets only partially mapped: {len(applied)}/{len(patches)}")\n            matched = len(applied)\n        options = dict(result.model_options)\n        entries = list(options.get("fast_h3_loras", []))\n        entries.append({"name": lora_name, "sha256": digest.hexdigest(),\n                        "strength_model": float(strength_model), "mapped_targets": matched})\n        options["fast_h3_loras"] = entries\n        result.model_options = options\n        return (result,)\n\n\nclass FastH3ReportAndTrim:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {"required": {\n            "model": ("MODEL",), "sigmas": ("SIGMAS",),\n            "images": ("IMAGE",), "audio": ("AUDIO",),\n            "duration_seconds": ("FLOAT", {"default": 5.0, "min": 5.0, "max": 15.0, "step": 1.0}),\n        }}\n\n    RETURN_TYPES = ("IMAGE", "AUDIO", "STRING")\n    RETURN_NAMES = ("images", "audio", "report")\n    FUNCTION = "report"\n    CATEGORY = "FastH3/output"\n    OUTPUT_NODE = True\n\n    def report(self, model, sigmas, images, audio, duration_seconds):\n        import torch\n        frames = round(duration_seconds * 24)\n        if images.shape[0] < frames:\n            raise ValueError("FastH3 decoded fewer video frames than the requested duration")\n        waveform = audio["waveform"]\n        rate = int(audio["sample_rate"])\n        samples = round(duration_seconds * rate)\n        if rate != 32000 or waveform.shape[1] != 2 or waveform.shape[-1] < samples:\n            raise ValueError("FastH3 requires complete 32 kHz stereo audio")\n        candidates = [value for value in model.model_options.get("transformer_options", {}).values()\n                      if hasattr(value, "last_summary") and hasattr(value, "kernel_calls")]\n        if len(candidates) != 1 or candidates[0].last_summary is None:\n            raise RuntimeError("The learned VSA runtime did not produce an execution summary")\n        vsa = dict(candidates[0].last_summary)\n        if (vsa["kernel_calls"] != 200 or vsa["dense_fallbacks"] != 0 or vsa["errors"] != 0):\n            raise RuntimeError(f"Expected 50 blocks × 4 native VSA forwards with zero fallbacks: {vsa}")\n        if len(sigmas) != 5 or float(sigmas[-1]) != 0:\n            raise ValueError("FastH3 must execute the four trained forwards plus the zero endpoint")\n        receipt = {\n            "backend": "comfyui_fasth3", "mode": "t2va", "vsa": vsa,\n            "timesteps": [999, 749, 500, 250], "sigmas": sigmas.tolist(),\n            "steps": 4, "sampler": "euler", "video_shift": 12.0, "audio_shift": 3.0,\n            "width": int(images.shape[2]), "height": int(images.shape[1]),\n            "model_frames": int(images.shape[0]), "output_frames": frames,\n            "fps": 24, "duration_seconds": duration_seconds,\n            "audio_sample_rate": rate, "audio_channels": 2,\n            "loras": model.model_options.get("fast_h3_loras", []),\n            "peak_torch_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),\n            "peak_torch_reserved_gib": round(torch.cuda.max_memory_reserved() / 2**30, 3),\n        }\n        reference = model.model_options.get("fast_h3_reference")\n        if reference is not None:\n            if len(reference["forward_seconds"]) != 4 or len(reference["layouts"]) != 4:\n                raise RuntimeError("Experimental Ref2VA did not execute four traced reference-conditioned forwards")\n            receipt.update(mode="experimental_ref2va_ood", reference=json.loads(json.dumps(reference.data)),\n                           supported_by_distillation=False)\n        trimmed_audio = dict(audio, waveform=waveform[..., :samples])\n        return {"result": (images[:frames], trimmed_audio, json.dumps(receipt)),\n                "ui": {"fast_h3": [receipt]}}\n\n\nNODE_CLASS_MAPPINGS = {\n    "FastH3SampleRequest": FastH3SampleRequest,\n    "FastH3ExperimentalReference": FastH3ExperimentalReference,\n    "FastH3ExactSigmas": FastH3ExactSigmas,\n    "FastH3CheckedLoraLoader": FastH3CheckedLoraLoader,\n    "FastH3ReportAndTrim": FastH3ReportAndTrim,\n}\n\nNODE_DISPLAY_NAME_MAPPINGS = {\n    "FastH3SampleRequest": "FastH3 新推論請求（保留 seed）",\n    "FastH3ExperimentalReference": "FastH3 人物參照（未蒸餾／實驗）",\n    "FastH3ExactSigmas": "FastH3 精確時間步 [999,749,500,250]",\n    "FastH3CheckedLoraLoader": "FastH3 相容 LoRA 載入",\n    "FastH3ReportAndTrim": "FastH3 VSA 執行紀錄與影音裁切",\n}\n', 'fast_h3_client.py': '"""Small standard-library client for an already running ComfyUI server."""\n\nimport filecmp\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\nfrom urllib.error import HTTPError, URLError\nfrom urllib.parse import quote, urlencode, urlsplit\nfrom urllib.request import Request, urlopen\nimport uuid\n\n\nclass GraphValidationError(ValueError):\n    """A graph cannot satisfy the server\'s advertised node schema."""\n\n\nclass ComfyHTTPError(RuntimeError):\n    def __init__(self, status, payload):\n        self.status, self.payload = status, payload\n        super().__init__(f"ComfyUI HTTP {status}: {payload}")\n\n\nclass PromptExecutionError(RuntimeError):\n    def __init__(self, prompt_id, history):\n        self.prompt_id, self.history = prompt_id, history\n        super().__init__(f"ComfyUI execution failed for prompt {prompt_id}")\n\n\nclass PromptTimeoutError(TimeoutError):\n    def __init__(self, prompt_id):\n        self.prompt_id = prompt_id\n        super().__init__(f"Timed out waiting for prompt {prompt_id}; it may still be running")\n\n\nclass ComfyClient:\n    def __init__(self, base_url="http://127.0.0.1:8188", timeout=30):\n        parsed = urlsplit(base_url)\n        if (parsed.scheme not in ("http", "https") or not parsed.hostname\n                or parsed.username or parsed.password or parsed.query or parsed.fragment):\n            raise ValueError("base_url must be an HTTP(S) server URL without credentials or query")\n        if timeout <= 0:\n            raise ValueError("timeout must be positive")\n        self.base_url, self.timeout = base_url.rstrip("/"), timeout\n\n    def _open(self, path, data=None, timeout=None):\n        if not path.startswith("/") or path.startswith("//"):\n            raise ValueError("API path must start with one slash")\n        body = None if data is None else json.dumps(data).encode("utf-8")\n        headers = {} if body is None else {"Content-Type": "application/json"}\n        request = Request(self.base_url + path, data=body, headers=headers)\n        try:\n            return urlopen(request, timeout=self.timeout if timeout is None else timeout)\n        except HTTPError as error:\n            with error:\n                raw = error.read().decode("utf-8", errors="replace")\n            try:\n                payload = json.loads(raw)\n            except ValueError:\n                payload = raw\n            raise ComfyHTTPError(error.code, payload) from error\n\n    def get_json(self, path):\n        with self._open(path) as response:\n            return json.load(response)\n\n    def post_json(self, path, data):\n        # A lost response does not imply the prompt was not queued. Never retry.\n        with self._open(path, data) as response:\n            return json.load(response)\n\n    def object_info(self):\n        return self.get_json("/object_info")\n\n    @staticmethod\n    def _input_specs(declared, inputs, prefix=""):\n        """Expand current ComfyUI DynamicCombo fields from their selected keys."""\n        specs, required = {}, set()\n        for group in ("required", "optional"):\n            for name, spec in declared.get(group, {}).items():\n                identity = prefix + name\n                specs[identity] = spec\n                if group == "required":\n                    required.add(identity)\n                if spec[0] == "COMFY_DYNAMICCOMBO_V3" and identity in inputs:\n                    options = spec[1].get("options", [])\n                    selected = next((option for option in options if option["key"] == inputs[identity]), None)\n                    if selected is None:\n                        raise GraphValidationError(f"Input {identity}: invalid dynamic combo selection")\n                    nested, needed = ComfyClient._input_specs(selected.get("inputs", {}), inputs, identity + ".")\n                    specs.update(nested)\n                    required.update(needed)\n        return specs, required\n\n    def validate_graph(self, graph):\n        """Check advertised inputs; /prompt still performs native validation."""\n        schemas = self.object_info()\n        if not isinstance(graph, dict) or not graph:\n            raise GraphValidationError("Graph must be a nonempty API node dictionary")\n        for node_id, node in graph.items():\n            if not isinstance(node_id, str) or not isinstance(node, dict):\n                raise GraphValidationError("API node IDs must be strings and nodes dictionaries")\n            class_type = node.get("class_type")\n            if not isinstance(class_type, str) or class_type not in schemas:\n                raise GraphValidationError(f"Node {node_id}: unknown class_type {class_type!r}")\n            inputs = node.get("inputs")\n            if not isinstance(inputs, dict):\n                raise GraphValidationError(f"Node {node_id}: inputs must be a dictionary")\n            declared = schemas[class_type].get("input", {})\n            specs, required = self._input_specs(declared, inputs)\n            missing = required - inputs.keys()\n            if missing:\n                raise GraphValidationError(f"Node {node_id}: missing inputs {sorted(missing)}")\n            for name, value in inputs.items():\n                label = f"Node {node_id} input {name}"\n                spec = specs.get(name)\n                if isinstance(value, list) and len(value) == 2 and isinstance(value[0], str):\n                    source, slot = value\n                    source_node = graph.get(source)\n                    source_class = source_node.get("class_type") if isinstance(source_node, dict) else None\n                    source_schema = schemas.get(source_class, {}) if isinstance(source_class, str) else {}\n                    outputs = source_schema.get("output", [])\n                    if source_node is None or type(slot) is not int or not 0 <= slot < len(outputs):\n                        raise GraphValidationError(f"{label}: invalid link {value!r}")\n                    continue\n                if not spec:\n                    continue  # Hidden/custom dynamic inputs are validated by ComfyUI.\n                kind = spec[0]\n                options = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}\n                if isinstance(kind, list):\n                    valid = value in kind\n                elif kind == "COMBO":\n                    choices = options.get("options", [])\n                    valid = (isinstance(value, list) and all(item in choices for item in value)\n                             if options.get("multiselect", False) else value in choices)\n                elif kind == "COMFY_DYNAMICCOMBO_V3":\n                    valid = any(option["key"] == value for option in options.get("options", []))\n                elif kind == "INT":\n                    valid = type(value) is int\n                elif kind == "FLOAT":\n                    valid = type(value) in (int, float) and math.isfinite(value)\n                elif kind == "BOOLEAN":\n                    valid = type(value) is bool\n                elif kind == "STRING":\n                    valid = isinstance(value, str)\n                else:\n                    valid = False  # Typed objects must arrive through links.\n                if not valid:\n                    raise GraphValidationError(f"{label}: invalid value for {kind!r}")\n                if kind in ("INT", "FLOAT") and any(\n                    key in options and compare(value, options[key])\n                    for key, compare in (("min", lambda a, b: a < b), ("max", lambda a, b: a > b))\n                ):\n                    raise GraphValidationError(f"{label}: value outside advertised range")\n        return graph\n\n    def queue_prompt(self, graph, prompt_id=None, extra_data=None):\n        self.validate_graph(graph)\n        identity = str(uuid.uuid4()) if prompt_id is None else str(prompt_id)\n        payload = {"prompt": graph, "prompt_id": identity}\n        if extra_data is not None:\n            payload["extra_data"] = extra_data\n        result = self.post_json("/prompt", payload)\n        if result.get("error") or result.get("node_errors") or not result.get("prompt_id"):\n            raise ComfyHTTPError(200, result)\n        return result\n\n    def _wait_json(self, path, prompt_id, deadline):\n        remaining = deadline - time.monotonic()\n        if remaining <= 0:\n            raise PromptTimeoutError(prompt_id)\n        try:\n            with self._open(path, timeout=min(self.timeout, remaining)) as response:\n                return json.load(response)\n        except TimeoutError as error:\n            raise PromptTimeoutError(prompt_id) from error\n        except URLError as error:\n            if isinstance(error.reason, TimeoutError):\n                raise PromptTimeoutError(prompt_id) from error\n            raise\n\n    def wait_prompt(self, prompt_id, timeout=1800, poll_seconds=5, on_status=None):\n        if timeout < 0 or poll_seconds <= 0:\n            raise ValueError("timeout must be nonnegative and poll_seconds positive")\n        started = time.monotonic()\n        deadline = started + timeout\n        while True:\n            history = self._wait_json("/history/" + quote(str(prompt_id), safe=""),\n                                      prompt_id, deadline).get(str(prompt_id))\n            status = history.get("status", {}) if history else {}\n            messages = status.get("messages", [])\n            failed = status.get("status_str") in ("error", "failed", "interrupted") or any(\n                message and message[0] in ("execution_error", "execution_interrupted", "execution_failed")\n                for message in messages\n            )\n            if failed:\n                raise PromptExecutionError(prompt_id, history)\n            complete = status.get("completed") is True and status.get("status_str") == "success"\n            state = "completed" if complete else "unknown"\n            if not complete:\n                queue = self._wait_json("/queue", prompt_id, deadline)\n                for key, value in (("queue_running", "running"), ("queue_pending", "pending")):\n                    if any(len(entry) > 1 and str(entry[1]) == str(prompt_id) for entry in queue.get(key, [])):\n                        state = value\n                        break\n            if on_status:\n                on_status({"prompt_id": str(prompt_id), "state": state,\n                           "elapsed_seconds": time.monotonic() - started, "status": status})\n            if complete:\n                return history\n            remaining = deadline - time.monotonic()\n            if remaining <= 0:\n                raise PromptTimeoutError(prompt_id)\n            time.sleep(min(poll_seconds, remaining))\n\n    @staticmethod\n    def output_files(history):\n        files = []\n        for output in history.get("outputs", {}).values():\n            for key in ("videos", "gifs", "images"):\n                for item in output.get(key, []):\n                    filename = item.get("filename", "")\n                    if Path(filename).suffix.lower() in (".mp4", ".webm", ".mov"):\n                        entry = {"filename": filename, "subfolder": item.get("subfolder", ""),\n                                 "type": item.get("type", "output")}\n                        if entry not in files:\n                            files.append(entry)\n        return files\n\n    def download_output(self, item, path):\n        target = Path(path)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        query = urlencode({key: item.get(key, default) for key, default in\n                           (("filename", ""), ("subfolder", ""), ("type", "output"))})\n        fd, temporary = tempfile.mkstemp(prefix="." + target.name + ".", dir=target.parent)\n        try:\n            with os.fdopen(fd, "wb") as output, self._open("/view?" + query) as response:\n                shutil.copyfileobj(response, output)\n                output.flush()\n                os.fsync(output.fileno())\n            # Hard-link publication is atomic and cannot replace an existing file.\n            # Unlike os.replace(), it also preserves a target created concurrently.\n            try:\n                os.link(temporary, target)\n            except FileExistsError:\n                if not filecmp.cmp(temporary, target, shallow=False):\n                    raise FileExistsError(f"Output already exists with different bytes: {target}")\n            return target\n        finally:\n            os.unlink(temporary)\n', 'fast_h3_lock.json': '{\n  "schema_version": 1,\n  "checked_date": "2026-10-09",\n  "gpu": "L4",\n  "comfyui": {\n    "repo": "https://github.com/Comfy-Org/ComfyUI.git",\n    "revision": "a4b5a045e56fc334903db8457b728b64e006119c"\n  },\n  "vsa_pack": {\n    "repo": "https://github.com/exportAnything/ComfyUI-FastH3-VSA.git",\n    "revision": "c1798183d89f7143d021c16507e4bb7ac0b3decc"\n  },\n  "comfy_kitchen": "0.2.37",\n  "assets": [\n    {\n      "repo": "Kijai/MiniMax-H3-experimental",\n      "revision": "d8023be02fefbb3633b0cd335c3879f91177299d",\n      "file": "minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors",\n      "size": 22898594920,\n      "sha256": "7221ae65d78780354d51e5048d29728d9f1f8fb9baf50b1dd3df85f5101413d3",\n      "destination": "diffusion_models/minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors"\n    },\n    {\n      "repo": "Comfy-Org/MiniMax-H3",\n      "revision": "e5eb578a89295337b8ff433a035929ce0279e0b6",\n      "file": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",\n      "size": 15687142551,\n      "sha256": "35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6",\n      "destination": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"\n    },\n    {\n      "repo": "Kijai/MiniMax-H3-experimental",\n      "revision": "d8023be02fefbb3633b0cd335c3879f91177299d",\n      "file": "minimax_h3_video_vae_int8_convrot.safetensors",\n      "size": 3171670912,\n      "sha256": "9bb2d96f218c76babd85e0611b85ca8fb330a90546c01a0005e8a58a59593410",\n      "destination": "vae/minimax_h3_video_vae_int8_convrot.safetensors"\n    },\n    {\n      "repo": "Comfy-Org/MiniMax-H3",\n      "revision": "e5eb578a89295337b8ff433a035929ce0279e0b6",\n      "file": "vae/minimax_h3_audio_vae_fp32.safetensors",\n      "size": 605254808,\n      "sha256": "8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48",\n      "destination": "vae/minimax_h3_audio_vae_fp32.safetensors"\n    }\n  ],\n  "source_checkpoint": "FastVideo FastH3 VSA DataFree 1300step (Kijai INT8 ConvRot conversion; article checkpoint, not v0.2 step2900)",\n  "timesteps": [\n    999,\n    749,\n    500,\n    250\n  ],\n  "video_shift": 12.0,\n  "audio_shift": 3.0,\n  "vsa": {\n    "keep_percent": 10.0,\n    "tile_size": 64,\n    "min_tokens": 8192,\n    "failure_mode": "error"\n  }\n}\n'}
# END EMBEDDED FAST H3 TOOLKIT

def using_fast_h3():
    if BACKEND not in ("FastH3 | ComfyUI L4 (T2VA)", "DiffSynth | Original H3 (FL2VA / Ref2VA)"):
        raise ValueError("請使用 BACKEND 選單中的引擎。")
    return BACKEND == "FastH3 | ComfyUI L4 (T2VA)"


def parse_fast_h3_loras():
    adapters = json.loads(FAST_H3_LORAS_JSON)
    if not isinstance(adapters, list) or not all(isinstance(item, dict) for item in adapters):
        raise ValueError("FAST_H3_LORAS_JSON 必須是 adapter 物件的 JSON 陣列。")
    return adapters


def get_fast_h3_runtime():
    # Literal sources make this notebook independent of a repository checkout.
    import sys
    toolkit = Path("/content/fast-h3-toolkit")
    toolkit.mkdir(parents=True, exist_ok=True)
    for filename, source in FAST_H3_SOURCES.items():
        destination = toolkit / filename
        if not destination.is_file() or destination.read_text() != source:
            destination.write_text(source)
    if str(toolkit) not in sys.path:
        sys.path.insert(0, str(toolkit))
    from fast_h3_runtime import FastH3Runtime
    global _FAST_H3_RUNTIME
    if globals().get("_FAST_H3_RUNTIME") is None:
        _FAST_H3_RUNTIME = FastH3Runtime(root="/content/fast-h3", port=8188)
    return _FAST_H3_RUNTIME



In [ ]:
# @title 2. 確認 L4 與實際資源
hardware = get_fast_h3_runtime().check_hardware()
print(json.dumps(hardware, indent=2))


In [ ]:
# @title 3. 安裝、下載與啟動本機 ComfyUI
FAST_H3_SETUP = get_fast_h3_runtime().setup(loras=parse_fast_h3_loras())
print(json.dumps(FAST_H3_SETUP, indent=2, ensure_ascii=False))


In [ ]:
# @title 4. 可選：Colab 本機連接埠預覽
SHOW_LOCAL_COMFYUI = False # @param {type:"boolean"}
if SHOW_LOCAL_COMFYUI:
    from google.colab import output
    output.serve_kernel_port_as_iframe(8188, height=700)


## B. 每片只執行這一區

改提示詞、尺寸、秒數與 seed，再產片。輸出資料夾與 metrics 由共用 runtime 記錄；檔案驗證通過後仍需觀看動作與聆聽音訊。


In [ ]:
# @title 5. 本片內容
# FastH3 T2VA inputs are independent of the native prompt and reference inputs below.
FAST_H3_PROMPT = "integrated_multimodal_description: [Shot 1] A single red toy train moves steadily from left to right across a wooden tabletop, its wheels visibly turning. Static camera, soft daylight, clear continuous movement. overall_soundscape: Small wheels rattle and click softly against the wooden tabletop. non_diegetic_music: N/A." # @param {type:"string"}
FAST_H3_WIDTH = 864 # @param {type:"integer"}
FAST_H3_HEIGHT = 480 # @param {type:"integer"}
FAST_H3_DURATION_SECONDS = 5 # @param {type:"integer"}
FAST_H3_SEED = 123456662 # @param {type:"integer"}
FAST_H3_RUN_NAME = "fast_h3_notebook" # @param {type:"string"}
# FastH3 renders at 24 fps; it does not consume native FIRST/LAST_IMAGE_PATH or REFERENCE_PATHS.


In [ ]:
# @title 6. 沿用 server 產片、驗證與預覽
import json
from pathlib import Path

def generate_fast_h3_clip():
    global OUTPUT_PATH, RUN_DIR, FAST_H3_RESULT
    runtime = get_fast_h3_runtime()
    adapters = parse_fast_h3_loras()
    runtime.ensure_ready(loras=adapters)
    FAST_H3_RESULT = runtime.generate(
        prompt=FAST_H3_PROMPT, width=FAST_H3_WIDTH, height=FAST_H3_HEIGHT,
        duration_seconds=FAST_H3_DURATION_SECONDS, seed=FAST_H3_SEED,
        loras=adapters, run_name=FAST_H3_RUN_NAME, timeout=1800)
    OUTPUT_PATH = Path(FAST_H3_RESULT["video_path"])
    RUN_DIR = Path(FAST_H3_RESULT["run_dir"])
    print(json.dumps(FAST_H3_RESULT, indent=2, ensure_ascii=False))
    from IPython.display import Video, display
    display(Video(str(OUTPUT_PATH), embed=True, width=720))
    return OUTPUT_PATH

generate_fast_h3_clip()


下載 `OUTPUT_PATH` 與 `RUN_DIR` 中要保留的紀錄，再停止 Colab runtime。

```python
from google.colab import files
files.download(str(OUTPUT_PATH))
```
